# Python 3.15 Data Engineering — Phase 6: Extended Stack Compatibility

**Author:** Dr. Ceasar Jackson Jr.  
**Platform:** macOS 26.5 ARM64  
**Python:** 3.15.0b1  
**Environment:** uv · `.venv`  

---

This notebook documents the Phase 6 extended stack compatibility findings interactively,
demonstrates the PyArrow cp315 wheel gap, and confirms the Docker workaround.

**Sections**
1. Compatibility matrix visualisation
2. PyArrow wheel gap — confirmed
3. Prefect stdlib breakage — confirmed
4. Dask partial compatibility — array vs. dataframe
5. Docker workaround — PyArrow via py314 container
6. Ecosystem blocker summary

## 1. Compatibility Matrix

In [ ]:
import sys, importlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print(f"Python: {sys.version.split()[0]}")

# Full compatibility matrix based on Phase 6 results
matrix = [
    # (package, category, status, blocker)
    ("numpy 2.4.6",        "Scientific",       "PASS",     "—"),
    ("pandas 3.0.3",       "DataFrame",        "PASS",     "—"),
    ("polars 1.41.2",      "DataFrame",        "PASS",     "—"),
    ("duckdb 1.5.3",       "Query engine",     "PASS",     "—"),
    ("sqlalchemy 2.0.50",  "ORM",              "PASS",     "—"),
    ("pydantic 2.13.4",    "Validation",       "PASS",     "—"),
    ("matplotlib 3.10.9",  "Visualisation",    "PASS",     "—"),
    ("plotly 6.7.0",       "Visualisation",    "PASS",     "—"),
    ("jupyterlab 4.5.7",   "Notebook",         "PASS",     "—"),
    ("pyarrow",            "Arrow/Parquet",    "INCOMPAT", "No cp315 wheels"),
    ("dask.dataframe",     "Distributed",      "INCOMPAT", "Runtime pyarrow dep"),
    ("dask.array/bag",     "Distributed",      "PASS",     "No pyarrow dep"),
    ("ray",                "Distributed",      "INCOMPAT", "No cp315 wheels"),
    ("delta lake",         "Table format",     "INCOMPAT", "pyarrow hard dep"),
    ("mlflow ≥ 2.17",      "ML lifecycle",     "INCOMPAT", "pyarrow hard dep"),
    ("prefect 3.7.x",      "Orchestration",    "INCOMPAT", "typing stdlib removal"),
    ("pyspark",            "Distributed SQL",  "UNTESTED", "Requires JDK"),
    ("apache airflow",     "Orchestration",    "UNTESTED", "Deferred"),
]

status_colors = {
    "PASS":     "#1D9E75",
    "INCOMPAT": "#BA7517",
    "UNTESTED": "#888780",
}

fig, ax = plt.subplots(figsize=(11, 7))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(matrix) + 1)
ax.axis("off")

ax.text(0.03, len(matrix) + 0.5, "Package",    fontsize=10, fontweight="bold", color="#0D1B2A")
ax.text(0.35, len(matrix) + 0.5, "Category",   fontsize=10, fontweight="bold", color="#0D1B2A")
ax.text(0.56, len(matrix) + 0.5, "Status",     fontsize=10, fontweight="bold", color="#0D1B2A")
ax.text(0.68, len(matrix) + 0.5, "Blocker",    fontsize=10, fontweight="bold", color="#0D1B2A")
ax.axhline(len(matrix) + 0.1, color="#0D1B2A", linewidth=1.2)

for i, (pkg, cat, status, blocker) in enumerate(reversed(matrix)):
    y = i + 0.5
    bg = "#F8F8F6" if i % 2 == 0 else "white"
    ax.fill_between([0, 1], [y - 0.4], [y + 0.4], color=bg, zorder=0)
    ax.text(0.03, y, pkg,     fontsize=9.5, va="center", color="#2C2C2A")
    ax.text(0.35, y, cat,     fontsize=9,   va="center", color="#888780")
    c = status_colors[status]
    ax.text(0.56, y, status,  fontsize=9,   va="center", color=c, fontweight="bold")
    ax.text(0.68, y, blocker, fontsize=8.5, va="center", color="#5F5E5A")

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in status_colors.items()]
ax.legend(handles=legend_patches, loc="lower right", framealpha=0.5, fontsize=9)
ax.set_title("Python 3.15 Data Engineering — Compatibility Matrix",
             fontsize=12, fontweight="bold", pad=10)

plt.tight_layout()
plt.savefig("../data/chart_compatibility_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_compatibility_matrix.png")

## 2. PyArrow cp315 Wheel Gap — Confirmed

In [ ]:
# Attempt pyarrow import and document the failure
try:
    import pyarrow as pa
    print(f"pyarrow {pa.__version__} — imported successfully (unexpected on Python 3.15 beta)")
except ImportError as e:
    print(f"❌ pyarrow ImportError (expected): {e}")
    print()
    print("Root cause:")
    print("  • No prebuilt cp315 wheels published on PyPI as of June 2026")
    print("  • Source build requires Rust compiler + C++ dependency chain")
    print("  • pkg_resources (removed in Python 3.15) referenced in legacy setup.py")
    print()
    print("Workaround: pyarrow-dataeng:py314 Docker container")
    print("Monitoring: https://github.com/apache/arrow")

## 3. Prefect stdlib Breakage — Confirmed

In [ ]:
# Demonstrate the typing.no_type_check_decorator removal
import typing

removed_symbols = [
    "no_type_check_decorator",  # Removed in 3.15 (deprecated since 3.13)
]

print(f"Python {sys.version.split()[0]} typing module symbol audit:")
print()
for sym in removed_symbols:
    present = hasattr(typing, sym)
    status = "✅ present" if present else "❌ REMOVED — Python 3.15 stdlib change"
    print(f"  typing.{sym:<30} {status}")

print()
print("Impact: Prefect 3.7.x imports typing.no_type_check_decorator at startup.")
print("Result: ImportError — cannot be worked around by installation strategy.")
print("Fix:    Prefect must remove the reference upstream.")
print("Track:  https://github.com/PrefectHQ/prefect")

# Confirm prefect fails
try:
    import prefect
    print(f"\nprefect {prefect.__version__} — imported (unexpected)")
except ImportError as e:
    print(f"\n❌ prefect ImportError (expected): {e}")

## 4. Dask Partial Compatibility

In [ ]:
# dask.array and dask.bag work; dask.dataframe requires pyarrow at runtime
import dask
print(f"dask {dask.__version__}")

# dask.array — functional
import dask.array as da
import numpy as np

arr = da.from_array(np.arange(1_000_000).reshape(1000, 1000), chunks=(250, 250))
result = (arr ** 2).mean().compute()
print(f"\ndask.array: (arange(1M)**2).mean() = {result:.1f}")
print("✅ dask.array — functional on Python 3.15")

# dask.dataframe — blocked by pyarrow
print()
try:
    import dask.dataframe as dd
    import pandas as pd
    df = dd.from_pandas(pd.DataFrame({"x": range(100)}), npartitions=2)
    print(f"dask.dataframe — functional (unexpected on Python 3.15)")
except ImportError as e:
    print(f"❌ dask.dataframe ImportError (expected): {e}")
    print("   dask.dataframe imports pyarrow at runtime — blocked by cp315 wheel gap.")
    print("   Use polars or dask.array as alternatives.")

## 5. Docker Workaround — PyArrow via py314 Container

In [ ]:
import subprocess, json

DOCKER_IMAGE = "pyarrow-dataeng:py314"

# Check image availability
check = subprocess.run(
    ["docker", "image", "inspect", DOCKER_IMAGE],
    capture_output=True, timeout=10
)

if check.returncode != 0:
    print(f"❌ Docker image {DOCKER_IMAGE} not found.")
    print("   Build with: cd docker_pyarrow_lab && docker build -t pyarrow-dataeng:py314 .")
else:
    print(f"✅ Docker image {DOCKER_IMAGE} found")

    # Run a quick PyArrow smoke test inside the container
    script = """
import pyarrow as pa
import pyarrow.parquet as pq
import sys, json, tempfile, os

table = pa.table({"id": range(100), "value": [float(i) * 1.5 for i in range(100)]})
tmp = tempfile.mktemp(suffix=".parquet")
pq.write_table(table, tmp)
back = pq.read_table(tmp)
os.unlink(tmp)

print(json.dumps({
    "pyarrow_version": pa.__version__,
    "python_version":  sys.version.split()[0],
    "rows_written":    len(table),
    "rows_read":       len(back),
    "status":          "ok"
}))
"""

    result = subprocess.run(
        ["docker", "run", "--rm", "--interactive", DOCKER_IMAGE, "python3", "-"],
        input=script, capture_output=True, text=True, timeout=60
    )

    if result.returncode == 0:
        lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
        payload = json.loads(lines[-1])
        print(f"\nDocker container report:")
        print(f"  Python version  : {payload['python_version']}")
        print(f"  PyArrow version : {payload['pyarrow_version']}")
        print(f"  Rows written    : {payload['rows_written']}")
        print(f"  Rows read back  : {payload['rows_read']}")
        print(f"  Status          : {payload['status']}")
        print("\n✅ Docker PyArrow workaround — operational")
    else:
        print(f"❌ Docker run failed: {result.stderr[:300]}")

## 6. Ecosystem Blocker Summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

blockers = {
    "PyArrow cp315\nwheels absent": [
        "pyarrow (direct)", "dask.dataframe",
        "delta lake", "mlflow ≥ 2.17", "ray (separate)"
    ],
    "Python 3.15\nstdlib removal": [
        "prefect 3.7.x",
        "(typing.no_type_check_decorator)",
    ],
    "Untested /\nnot installed": [
        "pyspark (needs JDK)",
        "apache airflow",
    ],
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ["#FAEEDA", "#FCEBEB", "#F1EFE8"]
border_colors = ["#BA7517", "#A32D2D", "#888780"]
title_colors  = ["#633806", "#791F1F", "#444441"]

for ax, (title, items), bg, bc, tc in zip(
        axes, blockers.items(), colors, border_colors, title_colors):
    ax.set_facecolor(bg)
    for spine in ax.spines.values():
        spine.set_edgecolor(bc)
        spine.set_linewidth(2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.92, title, transform=ax.transAxes,
            ha="center", va="top", fontsize=11, fontweight="bold", color=tc)
    for i, item in enumerate(items):
        ax.text(0.5, 0.72 - i * 0.18, f"• {item}",
                transform=ax.transAxes, ha="center", va="top",
                fontsize=9, color="#2C2C2A")

fig.suptitle("Python 3.15 Ecosystem Blockers — Root Cause Grouping",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../data/chart_ecosystem_blockers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to data/chart_ecosystem_blockers.png")

print()
print("Key insight: the single largest blocker is PyArrow cp315 wheel absence.")
print("Resolving it unblocks: dask.dataframe, delta lake, mlflow.")
print("Prefect requires a separate upstream code fix — independent of PyArrow.")